# 01.3 · Pandas: Series, DataFrames, reading files, inspecting data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/03-pandas-basics/lesson.ipynb)

Read the [README](README.md) first. Here you build the five-row table, run the inspection checklist and match every number to your by-hand answers. Then the same checklist on 541,910 real rows.

## 1. The five-row table

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Invoice":     [536365, 536365, 536365, 536366, 536367],
    "StockCode":   ["85123A", "71053", "84406B", "22633", "84879"],
    "Quantity":    [6, 6, np.nan, 6, 32],
    "Price":       [2.55, 3.39, 2.75, 1.85, 1.69],
    "Customer ID": [17850, 17850, 17850, np.nan, 13047],
    "Country":     ["United Kingdom"] * 4 + ["France"],
})
df

,Invoice,StockCode,Quantity,Price,Customer ID,Country
0,536365,85123A,6.0,2.55,17850.0,United Kingdom
1,536365,71053,6.0,3.39,17850.0,United Kingdom
2,536365,84406B,NaN,2.75,17850.0,United Kingdom
3,536366,22633,6.0,1.85,NaN,United Kingdom
4,536367,84879,32.0,1.69,13047.0,France


In [2]:
print("shape  :", df.shape)
print("columns:", list(df.columns))
print("index  :", list(df.index))

shape  : (5, 6)
columns: ['Invoice', 'StockCode', 'Quantity', 'Price', 'Customer ID', 'Country']
index  : [0, 1, 2, 3, 4]


## 2. A column is a Series, and a Series wraps a NumPy array

In [3]:
q = df["Quantity"]
print(type(q))
print(q)
print("underlying array:", q.to_numpy(), "| dtype", q.dtype)

<class 'pandas.Series'>
0     6.0
1     6.0
2     NaN
3     6.0
4    32.0
Name: Quantity, dtype: float64
underlying array: [ 6.  6. nan  6. 32.] | dtype float64


`Quantity` was whole numbers, but one is missing, so Pandas stored it as `float64` (`NaN` is a float). Everything from 01.1 and 01.2 works on a Series:

In [4]:
print("vectorised :", (df["Quantity"] * df["Price"]).to_numpy())
print("mask       :", (df["Quantity"] > 6).to_numpy())
print("select     :", df["Price"][df["Quantity"] > 6].to_numpy())

vectorised : [15.3  20.34   nan 11.1  54.08]
mask       : [False False False False  True]
select     : [1.69]


## 3. The inspection checklist

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Invoice      5 non-null      int64  
 1   StockCode    5 non-null      str    
 2   Quantity     4 non-null      float64
 3   Price        5 non-null      float64
 4   Customer ID  4 non-null      float64
 5   Country      5 non-null      str    
dtypes: float64(3), int64(1), str(2)
memory usage: 461.0 bytes


Compare with your by-hand table: Quantity and Customer ID have 4 non-null of 5; StockCode and Country are text (`str` here; older Pandas versions print `object`).

In [6]:
df.describe()

,Invoice,Quantity,Price,Customer ID
count,5.000000,4.0,5.000000,4.00
mean,536365.600000,12.5,2.446000,16649.25
std,0.894427,13.0,0.693022,2401.50
min,536365.000000,6.0,1.690000,13047.00
25%,536365.000000,6.0,1.850000,16649.25
50%,536365.000000,6.0,2.550000,17850.00
75%,536366.000000,12.5,2.750000,17850.00
max,536367.000000,32.0,3.390000,17850.00


Quantity: count 4, mean 12.5, min 6, max 32, median (50%) 6, std 13, as calculated by hand. (Pandas' 75% is 12.5 rather than our rough 19 because it interpolates between sorted values; both are legitimate conventions.) The mean is double the median: one bulk order.

In [7]:
print(df.isna().sum())
print()
print("distinct invoices:", df["Invoice"].nunique())
print(df["Country"].value_counts())

Invoice        0
StockCode      0
Quantity       1
Price          0
Customer ID    1
Country        0
dtype: int64

distinct invoices: 3
Country
United Kingdom    4
France            1
Name: count, dtype: int64


In [8]:
df.describe(include=["object", "str"])     # text columns: count, distinct, most common, its frequency

,StockCode,Country
count,5,5
unique,5,2
top,85123A,United Kingdom
freq,1,4


## 4. Files: CSV out, CSV in

Reading and writing is one line each. `index=False` stops Pandas writing the 0…4 row labels as a column.

In [9]:
df.to_csv("five_rows.csv", index=False)
back = pd.read_csv("five_rows.csv")
back

,Invoice,StockCode,Quantity,Price,Customer ID,Country
0,536365,85123A,6.0,2.55,17850.0,United Kingdom
1,536365,71053,6.0,3.39,17850.0,United Kingdom
2,536365,84406B,NaN,2.75,17850.0,United Kingdom
3,536366,22633,6.0,1.85,NaN,United Kingdom
4,536367,84879,32.0,1.69,13047.0,France


In [10]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

cust = pd.read_csv(data_path("shopwave_customers_small.csv"))
print(cust.shape)
cust.head(3)

(20, 7)


,customer_id,visits_last_month,items_last_month,days_since_last,country,spend_next_month,churned
0,C001,2,2,55,UK,27.61,1
1,C002,11,8,44,FR,220.42,0
2,C003,10,6,22,DE,176.21,0


In [11]:
cust.describe().round(1)

,visits_last_month,items_last_month,days_since_last,spend_next_month,churned
count,20.0,20.0,20.0,20.0,20.0
mean,8.2,5.2,30.3,159.6,0.4
std,4.0,2.2,15.8,70.6,0.5
min,2.0,1.0,5.0,27.6,0.0
25%,6.0,4.5,20.0,119.4,0.0
50%,9.0,5.5,28.0,171.5,0.0
75%,11.0,7.0,41.8,209.8,1.0
max,14.0,8.0,58.0,268.1,1.0


## 5. The real data: 541,910 rows

`read_excel` is slow (about a minute); the cell saves a CSV next to the Excel file so every later lesson loads in a second.

In [12]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, time, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

xlsx = ensure_online_retail()
csv = xlsx.with_name("online_retail_2010_2011.csv")

if csv.exists():
    retail = pd.read_csv(csv, parse_dates=["InvoiceDate"])
else:
    t0 = time.perf_counter()
    retail = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    print(f"read_excel took {time.perf_counter() - t0:.0f} s")
    retail.to_csv(csv, index=False)            # save a fast copy for later lessons
print(retail.shape)

(541910, 8)


In [13]:
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
retail.sample(5, random_state=0)       # random rows: usually less tidy than the first five

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
174234,551776,22674,FRENCH TOILET SIGN BLUE METAL,3,2011-05-04 11:23:00,1.25,13268.0,United Kingdom
225132,556636,22560,TRADITIONAL MODELLING CLAY,1,2011-06-13 15:30:00,2.46,NaN,United Kingdom
118944,546502,84596B,SMALL DOLLY MIX DESIGN ORANGE BOWL,1,2011-03-14 11:46:00,0.42,16005.0,United Kingdom
45273,540255,22294,HEART FILIGREE DOVE SMALL,1,2011-01-05 16:50:00,2.51,NaN,United Kingdom
349689,C567518,22846,BREAD BIN DINER STYLE RED,-1,2011-09-20 16:28:00,16.95,NaN,EIRE


In [15]:
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  str           
 1   StockCode    541910 non-null  str           
 2   Description  540456 non-null  str           
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[us]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 59.5 MB


Read the non-null column: `Description` is missing in about 1,500 rows and `Customer ID` in about 135,000 (a quarter of all lines: guest checkouts). `Invoice` and `StockCode` are text because some values contain letters.

In [16]:
retail.describe().round(2)

,Quantity,InvoiceDate,Price,Customer ID
count,541910.00,541910,541910.00,406830.00
mean,9.55,2011-07-04 13:35:22.342307,4.61,15287.68
min,-80995.00,2010-12-01 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-28 11:34:00,1.25,13953.00
50%,3.00,2011-07-19 17:17:00,2.08,15152.00
75%,10.00,2011-10-19 11:27:00,4.13,16791.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


Three stories in one table:

- **Quantity** runs from −80,995 to +80,995, while the typical line (50%) is 3 units. Those extremes are one giant order and its cancellation.
- **Price** has a minimum of −11,062 and a maximum of 38,970, against a median of £2.08. Negative prices are accounting adjustments, not products.
- **Customer ID** `describe` is meaningless (it is an identifier, not a quantity), a reminder that `describe` summarises whatever is numeric, sensible or not.

In [17]:
print(retail.isna().sum())
print()
print(f"rows with no Customer ID: {retail['Customer ID'].isna().mean():.1%}")

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

rows with no Customer ID: 24.9%


In [18]:
print("distinct customers :", retail["Customer ID"].nunique())
print("distinct products  :", retail["StockCode"].nunique())
print("distinct countries :", retail["Country"].nunique())
print()
print(retail["Country"].value_counts().head(8))
print()
print(f"UK share of lines: {(retail['Country'] == 'United Kingdom').mean():.1%}")

distinct customers : 4372
distinct products  : 4070
distinct countries : 38

Country
United Kingdom    495478
Germany             9495
France              8558
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Name: count, dtype: int64

UK share of lines: 91.4%


## 6. What we found (write this down)

| Finding | Consequence for later |
|---|---|
| 25% of rows have no `Customer ID` | customer-level models (churn, spend) must drop or treat them separately (01.5) |
| Quantities and prices go negative | returns and adjustments; must be separated from sales (01.5) |
| A few lines are enormous | outliers that will dominate means and regression lines (01.8, Module 02) |
| `Invoice` starting with `C` means cancelled | a flag you need for every revenue report (01.4) |
| UK is >90% of rows | per-country analyses need care; small countries have few rows |

This table is the output of inspection: not numbers, but a to-do list.

## Quiz answers

1. A Series is a NumPy array plus an index (labels) and a name; `.to_numpy()` gives the array back.
2. Because the column contains at least one missing value, and `NaN` is a float; NumPy integer arrays cannot hold `NaN`.
3. `df.isna().sum()`.
4. The distribution is skewed or has outliers: a few large values pull the mean well above the typical (median) value.
5. The column holds text, or a mix of types; Pandas could not store it as a single numeric type. (Newer Pandas shows `str` for pure-text columns.)

**Next:** [01.4 Pandas: selecting, filtering, sorting, new columns](../04-pandas-select-filter/)